In [0]:
import time
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor


df = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/silver_2019"
)

feature_cols = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data_ml = assembler.transform(df) \
                   .select(
                       "features",
                       col("fare_amount").alias("label"),
                       "pickup_month"
                   )


train = data_ml.filter(col("pickup_month") <= 9)
test  = data_ml.filter(col("pickup_month") > 9)

print("Train Rows:", train.count())
print("Test Rows :", test.count())



strong_train = train.sample(0.3, seed=42)
strong_test  = test.sample(0.3, seed=42)

rf = RandomForestRegressor(
    numTrees=20,
    maxDepth=6,
    maxBins=32
)

strong_results = []

for partitions in [50, 100, 200]:

    spark.conf.set("spark.sql.shuffle.partitions", partitions)

    start = time.time()

    model = rf.fit(strong_train)
    _ = model.transform(strong_test)

    runtime = time.time() - start

    strong_results.append((partitions, runtime))

strong_df = spark.createDataFrame(
    strong_results,
    ["partitions", "training_time_sec"]
)

strong_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/taxi_data/strong_scaling"
)

print("Strong scaling complete")


weak_results = []

for fraction in [0.1, 0.3, 0.6]:

    sample_train = train.sample(fraction=fraction, seed=42)
    sample_test  = test.sample(fraction=fraction, seed=42)

    start = time.time()

    model = rf.fit(sample_train)
    _ = model.transform(sample_test)

    runtime = time.time() - start

    weak_results.append((fraction, runtime))

weak_df = spark.createDataFrame(
    weak_results,
    ["data_fraction", "training_time_sec"]
)

weak_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/taxi_data/weak_scaling"
)

print("Weak scaling complete")
print("Scalability analysis complete.")

In [0]:
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/strong_scaling"
).show()

spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/weak_scaling"
).show()